### import

In [ ]:
import os
import re

import numpy as np
import openvsp as vsp
import math

XSEC_CRV_TYPE = {
    vsp.XS_UNDEFINED : 'XS_UNDEFINED' ,
    vsp.XS_POINT : 'XS_POINT' ,
    vsp.XS_CIRCLE : 'XS_CIRCLE' ,
    vsp.XS_ELLIPSE : 'XS_ELLIPSE' ,
    vsp.XS_SUPER_ELLIPSE : 'XS_SUPER_ELLIPSE' ,
    vsp.XS_ROUNDED_RECTANGLE : 'XS_ROUNDED_RECTANGLE' ,
    vsp.XS_GENERAL_FUSE : 'XS_GENERAL_FUSE' ,
    vsp.XS_FILE_FUSE : 'XS_FILE_FUSE' ,
    vsp.XS_BICONVEX : 'XS_BICONVEX' ,
    vsp.XS_WEDGE : 'XS_WEDGE' ,
    vsp.XS_EDIT_CURVE : 'XS_EDIT_CURVE' ,
    vsp.XS_FOUR_SERIES : 'XS_FOUR_SERIES' ,
    vsp.XS_SIX_SERIES : 'XS_SIX_SERIES' ,
    vsp.XS_FILE_AIRFOIL : 'XS_FILE_AIRFOIL' ,
    vsp.XS_CST_AIRFOIL : 'XS_CST_AIRFOIL' ,
    vsp.XS_VKT_AIRFOIL : 'XS_VKT_AIRFOIL' ,
    vsp.XS_FOUR_DIGIT_MOD : 'XS_FOUR_DIGIT_MOD' ,
    vsp.XS_FIVE_DIGIT : 'XS_FIVE_DIGIT' ,
    vsp.XS_FIVE_DIGIT_MOD : 'XS_FIVE_DIGIT_MOD' ,
    vsp.XS_ONE_SIX_SERIES : 'XS_ONE_SIX_SERIES' ,
    vsp.XS_NUM_TYPES : 'XS_NUM_TYPES' ,
}

PARM_TYPE = { 
    vsp.PARM_DOUBLE_TYPE : 'double',
    vsp.PARM_INT_TYPE : 'int',
    vsp.PARM_BOOL_TYPE : 'boolean',
    vsp.PARM_FRACTION_TYPE : 'fraction',
    vsp.PARM_LIMITED_INT_TYPE : 'limited int',
    vsp.PARM_NOTEQ_TYPE : 'noteq',
    vsp.PARM_POWER_INT_TYPE : 'power int',
}

### パラメータ一覧

In [ ]:
def print_parm_discription(geom_id):
    parm_array = vsp.GetGeomParmIDs( geom_id )
    print(f'| {"parm_name":25s}', f'| {"type":<12s}', '| discription |')
    print('| --- | --- | --- |')
    for parm_id in parm_array:
        parm_name = vsp.GetParmName(parm_id)
        parm_type = vsp.GetParmType(parm_id)
        parm_discription = vsp.GetParmDescript(parm_id)

        print(f'| {parm_name:25s}', f'| {PARM_TYPE.get(parm_type, "Unknown"):12s}', '\t|', parm_discription.replace('Default Description', '-'), '|')

geom_id = vsp.AddGeom("WING")
vsp.SetGeomName(geom_id, "WingGeom")
print_parm_discription(geom_id)

In [ ]:
def print_xsec_parm_discription(geom_id):
    parm_array = vsp.GetXSecParmIDs( geom_id )
    print(f'| {"parm_name":25s}', f'| {"type":<12s}', '| discription |')
    print('| --- | --- | --- |')
    for parm_id in parm_array:
        parm_name = vsp.GetParmName(parm_id)
        parm_type = vsp.GetParmType(parm_id)
        parm_discription = vsp.GetParmDescript(parm_id)

        print(f'| {parm_name:25s}', f'| {PARM_TYPE.get(parm_type, "Unknown"):12s}', '\t|', parm_discription.replace('Default Description', '-'), '|')

geom_id = vsp.AddGeom("WING")
vsp.SetGeomName(geom_id, "WingGeom")
xsec_surf_id = vsp.GetXSecSurf(geom_id, 0)
xsec_id = vsp.GetXSec(xsec_surf_id, 1)
print_xsec_parm_discription(xsec_id)

### NACA4字系列

In [ ]:
# 新しいVSPモデルを作成
vsp.ClearVSPModel()

# 翼を追加してgeom_idを取得
geom_id = vsp.AddGeom("WING")
vsp.SetGeomName(geom_id, "WingGeom")

# WingGeomのxsec_surf_idを取得
xsec_surf_id = vsp.GetXSecSurf(geom_id, 0)

# 各断面のループ
for xsec_index in range(vsp.GetNumXSec(xsec_surf_id)):    

    # 断面タイプを変更する。
    crv_type = vsp.XS_FOUR_SERIES
    vsp.ChangeXSecShape(xsec_surf_id, xsec_index, crv_type)

    # 変更した断面の XSec ID を取得する。
    xsec_id = vsp.GetXSec(xsec_surf_id, xsec_index)

    # パラメータを設定 (例 : NACA2412)
    for parm_name, val in zip(['Camber', 'CamberLoc', 'ThickChord'], [0.02, 0.40, 0.12]):
        parm_id = vsp.GetXSecParm(xsec_id, parm_name)
        print('\t','parm_id', parm_id, parm_name)
        vsp.SetParmVal(parm_id, val)

    # スパン1, コード長1、後退角ゼロに設定
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Span'),       0.5)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Sweep'),      0)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Tip_Chord'),  1)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Root_Chord'), 1)

vsp.Update()
vsp.WriteVSPFile(f"XS_FOUR_SERIES.vsp3")

#     # 外部.datファイル
#     args = {'file' : f'{foil_name}.dat'}
#     crv_type = vsp.XS_FILE_AIRFOIL

# # crv_type = vsp.XS_VKT_AIRFOIL
# # args = {
# #      "npts" : 120,
# #      "alpha" : 0.0,
# #      "epsilon" : 0.1,
# #      "kappa" : 0.1,
# #      "tau" : 10.0,
# # }

# set_airfoil_xsec(xsec_surf_id, 0, crv_type, args) # Root
# set_airfoil_xsec(xsec_surf_id, 1, crv_type, args) # Tip


### NACA 5字系列

In [ ]:
# 新しいVSPモデルを作成
vsp.ClearVSPModel()

# 翼を追加してgeom_idを取得
geom_id = vsp.AddGeom("WING")
vsp.SetGeomName(geom_id, "WingGeom")

# WingGeomのxsec_surf_idを取得
xsec_surf_id = vsp.GetXSecSurf(geom_id, 0)

# 各断面のループ
for xsec_index in range(vsp.GetNumXSec(xsec_surf_id)):    

    # 断面タイプを変更する。
    crv_type = vsp.XS_FIVE_DIGIT
    vsp.ChangeXSecShape(xsec_surf_id, xsec_index, crv_type)

    # 変更した断面の XSec ID を取得する。
    xsec_id = vsp.GetXSec(xsec_surf_id, xsec_index)
    print(xsec_id)

    # パラメータを設定 (例 : NACA2412)
    for parm_name, val in zip(['IdealCl', 'CamberLoc', 'ThickChord'], [0.3, 0.25, 0.12]):
        parm_id = vsp.GetXSecParm(xsec_id, parm_name)
        print('\t','parm_id', parm_id, parm_name)
        vsp.SetParmVal(parm_id, val)

    # スパン1, コード長1、後退角ゼロに設定
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Span'),       0.5)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Sweep'),      0)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Tip_Chord'),  1)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Root_Chord'), 1)

vsp.Update()
vsp.WriteVSPFile(f"XS_FIVE_DIGIT.vsp3")



### NACA6系列

In [ ]:
# 新しいVSPモデルを作成
vsp.ClearVSPModel()

# 翼を追加してgeom_idを取得
geom_id = vsp.AddGeom("WING")
vsp.SetGeomName(geom_id, "WingGeom")

# WingGeomのxsec_surf_idを取得
xsec_surf_id = vsp.GetXSecSurf(geom_id, 0)

# 各断面のループ
for xsec_index in range(vsp.GetNumXSec(xsec_surf_id)):    

    # 断面タイプを変更する。
    crv_type = vsp.XS_SIX_SERIES
    vsp.ChangeXSecShape(xsec_surf_id, xsec_index, crv_type)

    # 変更した断面の XSec ID を取得する。
    xsec_id = vsp.GetXSec(xsec_surf_id, xsec_index)

    # パラメータを設定 (例 : NACA63-312)
    for parm_name, val in zip(['IdealCl', 'ThickChord'], [0.3, 0.12]):
        parm_id = vsp.GetXSecParm(xsec_id, parm_name)
        vsp.SetParmVal(parm_id, val)

    # スパン1, コード長1、後退角ゼロに設定
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Span'),       0.5)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Sweep'),      0)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Tip_Chord'),  1)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Root_Chord'), 1)

vsp.Update()
vsp.WriteVSPFile(f"XS_SIX_SERIES.vsp3")



### NACA16系列

In [ ]:
# 新しいVSPモデルを作成
vsp.ClearVSPModel()

# 翼を追加してgeom_idを取得
geom_id = vsp.AddGeom("WING")
vsp.SetGeomName(geom_id, "WingGeom")

# WingGeomのxsec_surf_idを取得
xsec_surf_id = vsp.GetXSecSurf(geom_id, 0)

# 各断面のループ
for xsec_index in range(vsp.GetNumXSec(xsec_surf_id)):    

    # 断面タイプを変更する。
    crv_type = vsp.XS_ONE_SIX_SERIES
    vsp.ChangeXSecShape(xsec_surf_id, xsec_index, crv_type)

    # 変更した断面の XSec ID を取得する。
    xsec_id = vsp.GetXSec(xsec_surf_id, xsec_index)

    # パラメータを設定 (例 : NACA16-312)
    for parm_name, val in zip(['IdealCl', 'ThickChord'], [0.3, 0.12]):
        parm_id = vsp.GetXSecParm(xsec_id, parm_name)
        vsp.SetParmVal(parm_id, val)

    # スパン1, コード長1、後退角ゼロに設定
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Span'),       0.5)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Sweep'),      0)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Tip_Chord'),  1)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Root_Chord'), 1)

vsp.Update()
vsp.WriteVSPFile(f"XS_ONE_SIX_SERIES.vsp3")



### 座標データ

In [ ]:
# 新しいVSPモデルを作成
vsp.ClearVSPModel()

# 翼を追加してgeom_idを取得
geom_id = vsp.AddGeom("WING")
vsp.SetGeomName(geom_id, "WingGeom")

# WingGeomのxsec_surf_idを取得
xsec_surf_id = vsp.GetXSecSurf(geom_id, 0)

# 各断面のループ
for xsec_index in range(vsp.GetNumXSec(xsec_surf_id)):    

    # 断面タイプを変更する。
    crv_type = vsp.XS_FILE_AIRFOIL
    vsp.ChangeXSecShape(xsec_surf_id, xsec_index, crv_type)

    # 変更した断面の XSec ID を取得する。
    xsec_id = vsp.GetXSec(xsec_surf_id, xsec_index)

    # パラメータを設定 (例 : 'E603.dat'を読み込む)
    vsp.ReadFileAirfoil(xsec_id, 'E603.dat')

    # スパン1, コード長1、後退角ゼロに設定
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Span'),       0.5)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Sweep'),      0)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Tip_Chord'),  1)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Root_Chord'), 1)

vsp.Update()
vsp.WriteVSPFile(f"XS_FILE_AIRFOIL.vsp3")



### CST翼型

In [ ]:
# 新しいVSPモデルを作成
vsp.ClearVSPModel()

# 翼を追加してgeom_idを取得
geom_id = vsp.AddGeom("WING")
vsp.SetGeomName(geom_id, "WingGeom")

# WingGeomのxsec_surf_idを取得
xsec_surf_id = vsp.GetXSecSurf(geom_id, 0)

# 各断面のループ
for xsec_index in range(vsp.GetNumXSec(xsec_surf_id)):    

    # 断面タイプを変更する。
    crv_type = vsp.XS_CST_AIRFOIL
    vsp.ChangeXSecShape(xsec_surf_id, xsec_index, crv_type)

    # 変更した断面の XSec ID を取得する。
    xsec_id = vsp.GetXSec(xsec_surf_id, xsec_index)

    # パラメータを設定 (例 : 'E603'相当のパラメータ)
    u_coefs = [  0.16987,  0.52115, -0.08066,  0.91091, -0.08964,  0.46638,  0.23017 ]
    l_coefs = [ -0.16987, -0.04044, -0.36415,  0.14310, -0.56062,  0.06706,  0.08466 ]
    vsp.SetUpperCST(xsec_id, len(u_coefs)-1, u_coefs)
    vsp.SetLowerCST(xsec_id, len(l_coefs)-1, l_coefs)

    # スパン1, コード長1、後退角ゼロに設定
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Span'),       0.5)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Sweep'),      0)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Tip_Chord'),  1)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Root_Chord'), 1)

vsp.Update()
vsp.WriteVSPFile(f"XS_CST_AIRFOIL.vsp3")



### VTK翼型

In [ ]:
# 新しいVSPモデルを作成
vsp.ClearVSPModel()

# 翼を追加してgeom_idを取得
geom_id = vsp.AddGeom("WING")
vsp.SetGeomName(geom_id, "WingGeom")

# WingGeomのxsec_surf_idを取得
xsec_surf_id = vsp.GetXSecSurf(geom_id, 0)

# 各断面のループ
for xsec_index in range(vsp.GetNumXSec(xsec_surf_id)):    

    # 断面タイプを変更する。（XS_FILE_AIRFOIL のときしか SetAirfoilPnts は使えない）
    crv_type = vsp.XS_FILE_AIRFOIL
    vsp.ChangeXSecShape(xsec_surf_id, xsec_index, crv_type)

    # 変更した断面の XSec ID を取得する。
    xsec_id = vsp.GetXSec(xsec_surf_id, xsec_index)

    npts     = 121
    alpha    = 2 * math.pi / 180.0  # degree→rad
    epsilon  = 0.1
    kappa    = 0.1
    tau      = 10 * math.pi / 180.0   # degree→rad

    # VKT 点列を取得
    xyz_airfoil = vsp.GetVKTAirfoilPnts(npts, alpha, epsilon, kappa, tau)

    # （オプション）numpy 配列に変換
    # xyz_np = np.array([[p.x(), p.y(), p.z()] for p in xyz_airfoil])

    # 上下面に分割
    half = npts // 2
    up_pnt_vec  = xyz_airfoil[:half][::-1]
    low_pnt_vec = xyz_airfoil[half:]

    vsp.SetAirfoilPnts(xsec_id, low_pnt_vec, up_pnt_vec)

    # スパン1, コード長1、後退角ゼロに設定
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Span'),       0.5)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Sweep'),      0)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Tip_Chord'),  1)
    vsp.SetParmValUpdate(vsp.GetXSecParm(xsec_id, 'Root_Chord'), 1)

vsp.Update()
vsp.WriteVSPFile(f"XS_VKT_AIRFOIL.vsp3")

